In [0]:
## In this layer we will have business logics

## loading data from silver layer

gold_customers = spark.table("silver_customers")
gold_transactions = spark.table("silver_transactions")
gold_accounts = spark.table("silver_accounts")
gold_logins = spark.table("silver_logins")

In [0]:
df = spark.sql("SHOW COLUMNS IN silver_customers")
display(df)

df = spark.sql("SHOW COLUMNS IN silver_transactions")
display(df)

df = spark.sql("SHOW COLUMNS IN silver_logins")
display(df)

df = spark.sql("SHOW COLUMNS IN silver_accounts")
display(df)


In [0]:
## joining tables , creating Customer 360 view

customer_360 = gold_customers.join(gold_accounts, "customer_id" , "left").join(gold_transactions, "account_id" , "left")
display(customer_360)

In [0]:
## ADD FRAUD SIGNALS

# High amount transaction = risk
# Fail login attempts = risk
# Multiple accounts = risk

from pyspark.sql.functions import *

high_val_txn = gold_transactions.filter(col("amount")>10000).groupBy("account_id").agg(count("*").alias("high_val_txn_cnt"))

display(high_val_txn)

In [0]:
# Fail login attempts = risk

fail_login_attempts = gold_logins.filter(col("login_status")=="Failed").groupBy("customer_id").agg(count("*").alias("fail_login_attempts_cnt"))

display(fail_login_attempts)


In [0]:
high_acct_cnt = gold_accounts.groupBy("customer_id").agg(count("*").alias("high_acct_cnt")).filter("high_acct_cnt>1")

display(high_acct_cnt)

In [0]:
# Join everything to customer view

customer_360 = customer_360.join(high_val_txn,"account_id","left").join(fail_login_attempts,"customer_id","left").join(high_acct_cnt,"customer_id","left")

display(customer_360)

In [0]:
customer_360 = customer_360.fillna(0,subset=["high_val_txn_cnt","fail_login_attempts_cnt","high_acct_cnt"])

display(customer_360)

In [0]:
## Now calculate risk factor by assigning risk rating for each

from pyspark.sql.functions import *


customer_360 = customer_360.withColumn(
    "risk_rating",
    (col("high_acct_cnt") * 1) +
    (col("fail_login_attempts_cnt") * 5) +
    (col("high_val_txn_cnt") * 10)
)

display(customer_360)


In [0]:
#Save value to table

customer_360.write.mode("overwrite").saveAsTable(
    "gold_customer_risk"
)